# Pressure-Performance Ranking — T20 Cricket (Fixed & Unified v2)
**IPL 2022–2026 + Full-Member T20Is (Including Super Overs)**

### Summary of 5 Fixes Applied:
1. **Baseline-Adjusted Pressure Uplift (`build_eb`)**: Fits WLS regression $\mathbb{E}[P \mid N] = \hat{\alpha} + \hat{\beta} N$ and evaluates $\Delta_i^{\text{adj}} = P_i - \hat{\beta}\max(N_i, 0)$ with baseline-informed prior $\hat{\beta}\min(N_i, 0)$, eliminating the negative-$N$ subsidy that previously inflated poor normal-overs performers.
2. **OOF Skill Target with Wickets & Extras (Step 3A)**: Replaces `y = runs_bat` with run-equivalent delivery value `y = approx_runs - 6.0 * is_wicket` across all non-bye/legbye deliveries, properly crediting strike bowlers and penalizing dismissals.
3. **Dynamic Innings-1 Par Lookup & All-Out Terminal Condition (`shift_state` & `terminal`)**: Recomputes `score_minus_par` dynamically via a 2D `(balls_left, wickets)` par grid on every counterfactual ball and enforces `balls_left_after = 0` when `wkts_after >= 10` in both innings.
4. **Unified Wide/No-Ball Counterfactual Expectation (`OUTS`)**: Incorporates `("E", r=1, w=0, bl_drop=0)` into `OUTS` so pre-ball expected $\Delta\text{WP}$ (`exp_dwp`) and `leverage` account for zero ball-drop on wides and no-balls.
5. **Super Overs Retained (`innings` 3 & 4)**: Keeps all 38 Super Over deliveries in Step 1 & Step 2 (`inn_total_balls = 6`, mapped survival resources `wkts_before = wkts_raw + 8`, `is_pressure = True`).

## Step 0 — Setup & Config

In [7]:
import gc, warnings, os
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

DATA = Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\data")
OUT  = Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\Analysis_v2")
OUT.mkdir(exist_ok=True)

# Verify
assert DATA.exists(), f"Data dir not found: {DATA}"
print("Data files:", sorted(x.name for x in DATA.glob("*.csv")))

try:
    import psutil
    def mem(): return f"{round(psutil.Process().memory_info().rss/1e6)} MB"
except ImportError:
    def mem(): return "(psutil not installed)"
print("RAM:", mem())

Data files: ['ball_by_ball.csv', 'competition_team_player_mapping.csv', 'competitions.csv', 'matches.csv', 'playing_eleven.csv', 'teams.csv']
RAM: 1135 MB


---
## Step 1 — Load & Clean

In [8]:
# ── ball_by_ball: keep all innings including Super Overs (innings 3, 4) ──
keep_cols = [
    "match_id", "innings", "overid", "innings_ball_num",
    "batting_team_id", "bowling_team_id",
    "striker_id", "striker_name", "bowler_id", "bowler_name",
    "runs", "is_wicket", "dismissal_type", "dismissed_player_id",
    "extras_type", "extras_2_runs",
    "pace_or_spin"
]
bbb = pd.read_csv(DATA / "ball_by_ball.csv", usecols=keep_cols)
bbb = bbb[bbb["innings"].isin([1, 2, 3, 4])].copy()
print(f"Ball-by-ball (incl. Super Overs): {bbb.shape}")

matches = pd.read_csv(DATA / "matches.csv", usecols=[
    "match_id", "comp_id", "match_date", "stage", "knock_out",
    "revised_target", "first_inn_revised_overs", "second_inn_revised_overs",
    "venue_id", "win_team_id"
])
comps = pd.read_csv(DATA / "competitions.csv", usecols=["comp_id", "match_format", "season"])

df = bbb.merge(matches, on="match_id", how="left").merge(comps, on="comp_id", how="left")
del bbb; gc.collect()
df = df.sort_values(["match_id", "innings", "innings_ball_num"]).reset_index(drop=True)

df["legal_ball"] = ~df["extras_type"].isin(["wide", "Overstep"])
df["extra_run"] = df["extras_type"].isin(["wide", "Overstep"]).astype(int)
df["approx_runs"] = df["runs"].fillna(0).astype(int) + df["extra_run"] + df["extras_2_runs"].fillna(0).astype(int) + df["extras_type"].isin(["byes","legbyes"]).astype(int)
df["runs_bat"] = df["runs"].fillna(0).astype(int)
df["striker_out"] = (df["is_wicket"].astype(bool)) & (df["dismissed_player_id"] == df["striker_id"])

df["bowler_runs"] = df["runs"].fillna(0).astype(int) + df["extra_run"] + df["extras_2_runs"].fillna(0).astype(int)
df.loc[df["extras_type"].isin(["byes", "legbyes"]), "bowler_runs"] = 0

non_bowler_dismissals = ["RunOut", "Retire Out", "Obstructing the field", "Obstracting"]
df["bowler_wkt"] = (df["is_wicket"].astype(bool)) & (~df["dismissal_type"].isin(non_bowler_dismissals))
df["is_t20i"] = (df["match_format"] == "T20I").astype(int)
df["is_super_over"] = df["innings"].isin([3, 4]).astype(int)
print(f"Merged shape: {df.shape} | Super Over balls: {(df['is_super_over']==1).sum()}")
print("✓ Step 1 — Load & Clean complete")


Ball-by-ball (incl. Super Overs): (200999, 17)
Merged shape: (200999, 37) | Super Over balls: 38
✓ Step 1 — Load & Clean complete


---
## Step 2 — Reconstruct Match State Per Ball

In [9]:
# ── Innings total balls (120 regular, 6 Super Over, or DLS-shortened) ──
df["inn_total_balls"] = np.where(df["is_super_over"] == 1, 6, 120)
r1 = df["first_inn_revised_overs"]; r2 = df["second_inn_revised_overs"]
ok1 = (df["innings"] == 1) & r1.between(1, 120)
ok2 = (df["innings"] == 2) & r2.between(1, 120)
df.loc[ok1, "inn_total_balls"] = r1[ok1].round().astype(int)
df.loc[ok2, "inn_total_balls"] = r2[ok2].round().astype(int)

g = df.groupby(["match_id", "innings"], sort=False)
df["score_before"] = g["approx_runs"].cumsum() - df["approx_runs"]
df["wkts_raw_before"] = g["is_wicket"].cumsum().astype(int) - df["is_wicket"].astype(int)
df["legal_before"] = g["legal_ball"].cumsum().astype(int) - df["legal_ball"].astype(int)
del g; gc.collect()

# Map Super Over wickets (0->8, 1->9, 2->10) so 2 wickets = all out
df["wkts_before"] = np.where(df["is_super_over"] == 1, (df["wkts_raw_before"] + 8).clip(upper=10), df["wkts_raw_before"].clip(upper=10))
df["balls_left_before"] = (df["inn_total_balls"] - df["legal_before"]).clip(lower=0)

df["score_after"] = df["score_before"] + df["approx_runs"]
df["wkts_after"]  = (df["wkts_before"] + df["is_wicket"].astype(int)).clip(upper=10)
df["legal_after"] = df["legal_before"] + df["legal_ball"].astype(int)
# Terminal all-out sets balls_left_after = 0 in both innings
df["balls_left_after"] = np.where(df["wkts_after"] >= 10, 0, (df["inn_total_balls"] - df["legal_after"]).clip(lower=0))

first_total = df[df["innings"] == 1].groupby("match_id")["approx_runs"].sum().rename("first_total")
so_total    = df[df["innings"] == 3].groupby("match_id")["approx_runs"].sum().rename("so_total")
df = df.merge(first_total, on="match_id", how="left").merge(so_total, on="match_id", how="left")

df["target"] = np.nan
df.loc[df["innings"] == 2, "target"] = df.loc[df["innings"] == 2, "first_total"] + 1
df.loc[(df["innings"] == 2) & df["revised_target"].notna(), "target"] = df.loc[(df["innings"] == 2) & df["revised_target"].notna(), "revised_target"]
df.loc[df["innings"] == 4, "target"] = df.loc[df["innings"] == 4, "so_total"] + 1

df["runs_needed_before"] = df["target"] - df["score_before"]
df["runs_needed_after"]  = df["target"] - df["score_after"]
df["eff_innings"] = np.where(df["innings"].isin([1, 3]), 1, 2)
df["bat_team_won"] = (df["win_team_id"] == df["batting_team_id"]).astype(float)
df.loc[df["win_team_id"].isna(), "bat_team_won"] = np.nan
df.to_pickle(OUT / "state.pkl")
print("✓ Step 2 — Match State (with Super Overs & All-Out Terminal State) complete")


✓ Step 2 — Match State (with Super Overs & All-Out Terminal State) complete


---
## Step 3 — Win-Probability Model (Leverage)

---
## Step 4 — Pressure Label

In [10]:
# ══════════════════════════════════════════════════════════════════════════
# NEW CELL: Out-of-Fold (OOF) Skill Model
# ══════════════════════════════════════════════════════════════════════════
import pandas as pd, numpy as np, gc
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from scipy.sparse import csr_matrix, hstack

OUT  = Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\Analysis_v2")
DATA = Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\data")

st = pd.read_pickle(OUT / "state.pkl")
st["phase"] = pd.cut(st["overid"], [0, 6, 15, 20], labels=[0, 1, 2]).astype(int)
if "season" not in st.columns:
    st = st.merge(pd.read_csv(DATA/"competitions.csv", usecols=["comp_id","season"]), on="comp_id", how="left")

# Use all legal balls with known outcomes for the skill fit
neutral_mask = st["extras_type"].fillna("none").isin(["byes", "legbyes"])
F = st[(~neutral_mask) & st["bat_team_won"].notna()].copy()
# FIX #2: Run-equivalent delivery value including wicket penalty (-6.0 runs) and extras
F["y"] = F["approx_runs"].astype(float) - 6.0 * F["is_wicket"].astype(float)
F["season_c"] = F["season"].astype(str)

striker_ids = sorted(F["striker_id"].unique())
bowler_ids  = sorted(F["bowler_id"].unique())
seasons     = sorted(F["season_c"].unique())
venues      = sorted(F["venue_id"].dropna().astype(int).unique())
sid = {v: i for i, v in enumerate(striker_ids)}
bid = {v: i for i, v in enumerate(bowler_ids)}

def design(frame):
    n = len(frame)
    def onehot(series, mapping):
        cols = series.map(mapping).fillna(0).astype(int).values
        return csr_matrix((np.ones(n, dtype=np.float32), (np.arange(n), cols)), shape=(n, len(mapping)))
    S = onehot(frame["striker_id"], sid); B = onehot(frame["bowler_id"], bid)
    P = pd.get_dummies(frame["phase"].astype(int), prefix="p").reindex(columns=[f"p_{i}" for i in range(3)], fill_value=0).values
    Se = pd.get_dummies(frame["season_c"], prefix="s").reindex(columns=[f"s_{s}" for s in seasons], fill_value=0).values
    V = pd.get_dummies(frame["venue_id"].astype(int), prefix="v").reindex(columns=[f"v_{v}" for v in venues], fill_value=0).values
    return hstack([S, B, csr_matrix(P), csr_matrix(Se), csr_matrix(V)]).tocsr()

X_all = design(F)
y = F["y"].values
groups = F["match_id"].values
ALPHA = 50.0

# Generate OOF Skills
oof_records = []
for tr, te in GroupKFold(5).split(X_all, y, groups):
    m = Ridge(alpha=ALPHA, fit_intercept=True, solver="lsqr").fit(X_all[tr], y[tr])
    c = m.coef_
    b_s = pd.Series(c[:len(striker_ids)], index=striker_ids); b_s -= b_s.mean()
    w_s = -pd.Series(c[len(striker_ids):len(striker_ids)+len(bowler_ids)], index=bowler_ids); w_s -= w_s.mean()
    
    te_matches = F.iloc[te][["match_id", "striker_id", "bowler_id"]].drop_duplicates()
    for _, row in te_matches.iterrows():
        oof_records.append({
            "match_id": row["match_id"], 
            "striker_id": row["striker_id"], 
            "bat_skill_oof": b_s.get(row["striker_id"], 0.0),
            "bowler_id": row["bowler_id"], 
            "bowl_skill_oof": w_s.get(row["bowler_id"], 0.0)
        })

oof_df = pd.DataFrame(oof_records)
# Since a match can have multiple bowlers, we need to separate batter skills and bowler skills to avoid duplication
bat_oof = oof_df[["match_id", "striker_id", "bat_skill_oof"]].drop_duplicates()
bowl_oof = oof_df[["match_id", "bowler_id", "bowl_skill_oof"]].drop_duplicates()

bat_oof.to_pickle(OUT / "bat_skills_oof.pkl")
bowl_oof.to_pickle(OUT / "bowl_skills_oof.pkl")

# Also save the full-data skill for the report sanity check
m_full = Ridge(alpha=ALPHA, fit_intercept=True, solver="lsqr").fit(X_all, y)
c = m_full.coef_
bat_full = pd.Series(c[:len(striker_ids)], index=striker_ids); bat_full -= bat_full.mean()
bowl_full = pd.Series(c[len(striker_ids):len(striker_ids)+len(bowler_ids)], index=bowler_ids); bowl_full -= bowl_full.mean()
mp = pd.read_csv(DATA/"competition_team_player_mapping.csv", usecols=["player_id","player_name"]).drop_duplicates("player_id")
report = pd.DataFrame({"player_id": mp["player_id"], "player_name": mp["player_name"]})
report["bat_skill_full"] = report["player_id"].map(bat_full).fillna(0.0)
report["bowl_skill_full"] = report["player_id"].map(bowl_full).fillna(0.0)
report.to_csv(OUT / "player_skills_full.csv", index=False)

print("✓ OOF skills saved. Ready for leakage-free WP model.")

✓ OOF skills saved. Ready for leakage-free WP model.


In [14]:
# ══════════════════════════════════════════════════════════════════════════
# Step 3, 4 & 5 (FINAL FIXED) — LEAKAGE-FREE WP MODEL + BASELINE-ADJUSTED EB
# ══════════════════════════════════════════════════════════════════════════
import gc, numpy as np, pandas as pd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier as HGB
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, log_loss

OUT = Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\Analysis_v2")
DATA = Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\data")

# ── 1. LOAD & PREP ──
df = pd.read_pickle(OUT / "state.pkl")
df["phase"] = pd.cut(df["overid"].clip(1, 20), [0, 6, 15, 20], labels=[0, 1, 2]).astype(int)

# Merge OOF skills (fitted with wicket & extra value in Step 3A)
bat_oof = pd.read_pickle(OUT / "bat_skills_oof.pkl")
bowl_oof = pd.read_pickle(OUT / "bowl_skills_oof.pkl")
df = df.merge(bat_oof, on=["match_id", "striker_id"], how="left")
df = df.merge(bowl_oof, on=["match_id", "bowler_id"], how="left")
df["batter_skill"] = df["bat_skill_oof"].fillna(0.0)
df["bowler_skill"] = df["bowl_skill_oof"].fillna(0.0)

# FIX #3A: Build 2D Par Score Grid by (balls_elapsed, wkts_before) so shift_state updates par dynamically
inn1_mask = (df["innings"] == 1) & (df["legal_before"].between(0, 120))
par_table = df[inn1_mask].groupby(["legal_before", "wkts_before"])["score_before"].mean().unstack()
par_table = par_table.reindex(index=range(0, 121), columns=range(0, 11)).interpolate(method="linear", axis=0).ffill().bfill()
for w in range(1, 11):
    par_table[w] = np.minimum(par_table[w], par_table[w-1] - 2.5)
PAR_GRID = par_table.values

def lookup_par(balls_left, wkts):
    b_elapsed = np.clip(120 - np.round(balls_left).astype(int), 0, 120)
    w_clip = np.clip(np.round(wkts).astype(int), 0, 10)
    return PAR_GRID[b_elapsed, w_clip]

df["par_score"] = lookup_par(df["balls_left_before"].values, df["wkts_before"].values)
df["score_minus_par"] = df["score_before"] - df["par_score"]

d0 = df[(df["balls_left_before"] > 0) & df["bat_team_won"].notna()].copy()
d0["pk"] = d0["is_t20i"].astype(int) * 3 + d0["phase"]

# FIX #4: Include ("E", r=1, w=0, bl_drop=0) for Wide/Overstep in outcome distribution
neutral_d0 = d0["extras_type"].fillna("none").isin(["byes", "legbyes"])
lg = d0[~neutral_d0].copy()
oc = np.where(
    ~lg["legal_ball"].astype(bool), "E",
    np.where(lg["is_wicket"].astype(bool), "W", lg["runs_bat"].clip(upper=6).replace({5: 4}).astype(str))
)
lg["o"] = oc

P_OUT_INN1 = lg[lg["eff_innings"] == 1].groupby("pk")["o"].value_counts(normalize=True).unstack().fillna(0)
lg2 = lg[lg["eff_innings"] == 2].copy()
lg2["rn_bin"] = pd.cut(lg2["runs_needed_before"], bins=[-100, 10, 20, 40, 1000], labels=["<10", "10-20", "20-40", ">40"])
lg2["pk_rich"] = lg2["phase"].astype(str) + "_" + lg2["rn_bin"].astype(str)
P_OUT_INN2 = lg2.groupby("pk_rich")["o"].value_counts(normalize=True).unstack().fillna(0)
OUTS = [("W", 0, 1, 1), ("E", 1, 0, 0)] + [(str(r), r, 0, 1) for r in [0, 1, 2, 3, 4, 6]]

# ── 2. WP MODEL ──
CFG = {
    1: dict(feat=["score_before", "wkts_before", "balls_left_before", "is_t20i", "knock_out", "score_minus_par", "batter_skill", "bowler_skill"],
            mk=lambda: HGB(max_depth=2, max_iter=120, learning_rate=0.05, random_state=42)),
    2: dict(feat=["runs_needed_before", "wkts_before", "balls_left_before", "is_t20i", "knock_out", "batter_skill", "bowler_skill"],
            mk=lambda: HGB(max_iter=250, learning_rate=0.06, max_leaf_nodes=15, min_samples_leaf=200, l2_regularization=1.0, random_state=0)),
}

# FIX #3B: Dynamic score_minus_par update + all-out terminal condition (wkts >= 10 -> balls_left = 0) in both innings
def shift_state(X, i, r, w, bl_drop):
    N = X.copy()
    N[:, 1] = np.clip(N[:, 1] + w, 0, 10)
    N[:, 2] = np.where(N[:, 1] >= 10, 0.0, np.clip(N[:, 2] - bl_drop, 0.0, None))
    if i == 1:
        N[:, 0] += r
        N[:, 5] = N[:, 0] - lookup_par(N[:, 2], N[:, 1])
    else:
        N[:, 0] -= r
    return N

def terminal(wp, N, i):
    if i == 2:
        wp = np.where(N[:, 0] <= 0, 1.0, wp)
        wp = np.where((N[:, 1] >= 10) | ((N[:, 2] <= 0) & (N[:, 0] > 0)), 0.0, wp)
    return wp

SAVE = ["match_id", "innings", "eff_innings", "is_super_over", "innings_ball_num", "striker_id", "striker_name", "bowler_id", "bowler_name",
        "runs_bat", "approx_runs", "legal_ball", "is_wicket", "striker_out", "bowler_wkt", "bowler_runs",
        "batting_team_id", "bowling_team_id", "extras_type", "overid", "phase", "wkts_before", "score_before", "runs_needed_before",
        "balls_left_before", "is_t20i", "wp_before", "wp_after", "leverage", "exp_dwp", "batter_skill", "bowler_skill"]

for i in (1, 2):
    d = d0[d0["eff_innings"] == i].copy()
    X = d[CFG[i]["feat"]].astype(float).values; y = d["bat_team_won"].astype(int).values
    wp_b, wp_a, lev, edw = (np.zeros(len(d)) for _ in range(4))
    for tr, te in GroupKFold(5).split(X, y, d["match_id"]):
        mdl = CFG[i]["mk"]().fit(X[tr], y[tr]); Xt = X[te]; T = d.iloc[te]
        p0 = mdl.predict_proba(Xt)[:, 1]
        Na = shift_state(Xt, i, T["approx_runs"].values, T["is_wicket"].astype(int).values, T["legal_ball"].astype(int).values)
        wp_a[te] = terminal(mdl.predict_proba(Na)[:, 1], Na, i)
        L = np.zeros(len(te)); E = np.zeros(len(te))
        for name, r, w, bl_d in OUTS:
            Nh = shift_state(Xt, i, r, w, bl_d); p1 = terminal(mdl.predict_proba(Nh)[:, 1], Nh, i)
            if i == 1:
                pr = T["pk"].map(P_OUT_INN1[name]).fillna(0.0).values
            else:
                rn_bin = pd.cut(T["runs_needed_before"], bins=[-100, 10, 20, 40, 1000], labels=["<10", "10-20", "20-40", ">40"])
                pk_rich = T["phase"].astype(str) + "_" + rn_bin.astype(str)
                pr = pk_rich.map(P_OUT_INN2[name]).fillna(P_OUT_INN2[name].mean()).values
            L += pr * np.abs(p1 - p0); E += pr * (p1 - p0)
        wp_b[te], lev[te], edw[te] = p0, L, E
    d["wp_before"], d["wp_after"], d["leverage"], d["exp_dwp"] = wp_b, wp_a, lev, edw
    print(f"Innings {i}: AUC {roc_auc_score(y, wp_b):.3f} | logloss {log_loss(y, wp_b):.3f}")
    d[SAVE].to_pickle(OUT / f"inn{i}_wp_skill.pkl")
    del d; gc.collect()

inn1 = pd.read_pickle(OUT / "inn1_wp_skill.pkl")
inn2 = pd.read_pickle(OUT / "inn2_wp_skill.pkl")
wp = pd.concat([inn1, inn2], ignore_index=True)
del inn1, inn2; gc.collect()

# FIX #5: Include all Super Over deliveries as high-pressure alongside top 25% leverage per innings
wp["is_pressure"] = wp["is_super_over"] == 1
for inn in (1, 2):
    mask = wp["eff_innings"] == inn
    q75 = wp.loc[mask, "leverage"].quantile(0.75)
    wp.loc[mask, "is_pressure"] = wp.loc[mask, "is_pressure"] | (wp.loc[mask, "leverage"] >= q75)

print(f"Pressure balls: {wp['is_pressure'].sum():,} (incl. {(wp['is_super_over']==1).sum()} Super Over balls) | Normal balls: {(~wp['is_pressure']).sum():,}")

# ── 3. SCORING & RANKING (FIX #1: Baseline-Adjusted WLS Empirical Bayes) ──
MIN_P_BAT, MIN_N_BAT = 100, 100
MIN_P_BOWL, MIN_N_BOWL = 120, 120
N_BOOT = 300
RNG = np.random.default_rng(7)
wv = wp.copy()

FULL = {"India","Australia","England","South Africa","New Zealand","Pakistan","Sri Lanka","Bangladesh","West Indies","Afghanistan","Zimbabwe","Ireland"}
_mm = pd.read_csv(DATA/"matches.csv", usecols=["match_id","comp_id"]).merge(pd.read_csv(DATA/"competitions.csv", usecols=["comp_id","comp_name"]), on="comp_id")
_tn = pd.read_csv(DATA/"teams.csv").drop_duplicates("team_id").set_index("team_id")["team_name"]
_t = wv.drop_duplicates("match_id")[["match_id","batting_team_id","bowling_team_id"]].merge(_mm, on="match_id")
_t["ipl"] = _t.comp_name.str.startswith("IPL"); _t["minor"] = _t.comp_name.str.contains("Emerging|Rising")
_t["both_full"] = _t.batting_team_id.map(_tn).isin(FULL) & _t.bowling_team_id.map(_tn).isin(FULL)
_keep = set(_t.loc[_t.ipl | (~_t.minor & _t.both_full), "match_id"])
wv = wv[wv.match_id.isin(_keep)].copy()

wv["dwp"] = (wv["wp_after"] - wv["wp_before"]) * 100
wv["excess"] = (wv["dwp"] - wv["exp_dwp"] * 100).fillna(0.0)
neutral = wv["extras_type"].fillna("none").isin(["byes","legbyes"])

bat0 = wv[wv["legal_ball"] & ~neutral].copy()
bat0["x"] = bat0["excess"]
bat0.loc[bat0["is_wicket"].astype(bool) & ~bat0["striker_out"].astype(bool), "x"] = 0.0
bat0["pid"] = bat0["striker_id"]

bow0 = wv[~neutral].copy()
bow0["x"] = -bow0["excess"]
bow0.loc[bow0["is_wicket"].astype(bool) & ~bow0["bowler_wkt"].astype(bool), "x"] = 0.0
bow0["pid"] = bow0["bowler_id"]

lab_map = wv.groupby("eff_innings")["leverage"].quantile(0.75).to_dict()
bat0["is_p"] = (bat0["is_super_over"] == 1) | (bat0["leverage"].values >= bat0["eff_innings"].map(lab_map).values)
bow0["is_p"] = (bow0["is_super_over"] == 1) | (bow0["leverage"].values >= bow0["eff_innings"].map(lab_map).values)

def build_eb(frame, wt_dict=None):
    if wt_dict is not None:
        w = frame["match_id"].map(wt_dict).fillna(0).values
        m = w > 0
        if m.sum() < 100: return None
        f = frame[m].copy()
        f["w"] = w[m]
    else:
        f = frame.copy()
        f["w"] = 1.0

    is_p = f["is_p"].values.astype(bool)
    x    = np.nan_to_num(f["x"].values, nan=0.0, posinf=0.0, neginf=0.0)
    w    = f["w"].values
    pid  = f["pid"].values

    p_mask = is_p
    n_mask = ~is_p

    p_df = pd.DataFrame({"pid": pid[p_mask], "w": w[p_mask], "wx": w[p_mask]*x[p_mask]}).groupby("pid").sum()
    n_df = pd.DataFrame({"pid": pid[n_mask], "w": w[n_mask], "wx": w[n_mask]*x[n_mask]}).groupby("pid").sum()

    T = p_df.join(n_df, how="inner", lsuffix="_p", rsuffix="_n").fillna(0)
    T.columns = ["n", "sx", "nn", "snx"]
    T = T[(T["n"] > 0) & (T["nn"] > 0)].copy()
    T["P"] = T["sx"] / T["n"]
    T["N"] = T["snx"] / T["nn"]

    xp_clip = np.clip(x[p_mask], -15.0, 15.0)
    xn_clip = np.clip(x[n_mask], -10.0, 10.0)
    var_p = np.average((xp_clip - np.average(xp_clip, weights=w[p_mask]))**2, weights=w[p_mask]) if p_mask.sum() > 0 else 1.0
    var_n = np.average((xn_clip - np.average(xn_clip, weights=w[n_mask]))**2, weights=w[n_mask]) if n_mask.sum() > 0 else 1.0

    v_p = var_p / T["n"]
    v_n = var_n / T["nn"]

    elig = T[(T["n"] >= 60) & (T["nn"] >= 60)]
    if len(elig) >= 10:
        w_reg = 1.0 / np.maximum(v_p.loc[elig.index].values + elig["P"].var(), 1e-6)
        w_reg = w_reg / w_reg.sum()
        x_e, y_e = elig["N"].values, elig["P"].values
        mx, my = np.sum(w_reg * x_e), np.sum(w_reg * y_e)
        vx = np.sum(w_reg * (x_e - mx) ** 2)
        beta_raw = np.sum(w_reg * (x_e - mx) * (y_e - my)) / max(vx, 1e-8)
        beta_hat = float(np.clip(beta_raw, 0.15, 0.50))
    else:
        beta_hat = 0.25

    N_ref = np.maximum(T["N"], 0.0)
    T["Delta"] = T["P"] - beta_hat * N_ref
    T["var_delta"] = v_p + (beta_hat ** 2) * v_n

    prior_mean = beta_hat * np.minimum(T["N"], 0.0)
    elig_d = T.loc[elig.index, "Delta"]
    elig_v = T.loc[elig.index, "var_delta"]
    tau2 = max(float(elig_d.var(ddof=1) - elig_v.median()), 0.08)

    B = tau2 / (tau2 + T["var_delta"])
    T["post_delta"] = prior_mean + B * (T["Delta"] - prior_mean)
    T["psd_delta"] = np.sqrt(B * T["var_delta"])
    T["lo90"] = T["post_delta"] - 1.645 * T["psd_delta"]
    T["hi90"] = T["post_delta"] + 1.645 * T["psd_delta"]
    return T

def rank_players(frame, namecol, min_p, min_n):
    T = build_eb(frame)
    if T is None: return pd.DataFrame()
    T = T[(T.n >= min_p) & (T.nn >= min_n)].copy()
    T["name"] = frame.groupby("pid")[namecol].agg(lambda s: s.mode()[0]).reindex(T.index)
    return T.sort_values("post_delta", ascending=False).reset_index()

bt_uplift = rank_players(bat0, "striker_name", MIN_P_BAT, MIN_N_BAT)
bo_uplift = rank_players(bow0, "bowler_name", MIN_P_BOWL, MIN_N_BOWL)

def fast_bootstrap(frame, min_p, min_n):
    ms = frame["match_id"].unique()
    top_hits = {}
    for b in range(N_BOOT):
        sampled = RNG.choice(ms, len(ms), replace=True)
        wt = pd.Series(sampled).value_counts().to_dict()
        T = build_eb(frame, wt)
        if T is None: continue
        T = T[(T.n >= min_p) & (T.nn >= min_n)]
        top10 = set(T.sort_values("post_delta", ascending=False).head(10).index)
        for pid in top10:
            top_hits[pid] = top_hits.get(pid, 0) + 1
        if (b+1) % 50 == 0:
            print(f"  boot {b+1}/{N_BOOT}")
    return (pd.Series(top_hits) / N_BOOT * 100).round(0)

print("\nBootstrapping batters...")
boot_bat = fast_bootstrap(bat0, MIN_P_BAT, MIN_N_BAT)
print("Bootstrapping bowlers...")
boot_bowl = fast_bootstrap(bow0, MIN_P_BOWL, MIN_N_BOWL)

bt_uplift = rank_players(bat0, "striker_name", MIN_P_BAT, MIN_N_BAT)
bo_uplift = rank_players(bow0, "bowler_name", MIN_P_BOWL, MIN_N_BOWL)

bt_uplift["boot_top10_pct"] = bt_uplift["pid"].map(boot_bat).fillna(0)
bo_uplift["boot_top10_pct"] = bo_uplift["pid"].map(boot_bowl).fillna(0)

cols = ["name","n","nn","P","N","Delta","post_delta","lo90","hi90","boot_top10_pct"]
print(f"\nEligible batters: {len(bt_uplift)} | bowlers: {len(bo_uplift)}")
print("\n" + "═"*70)
print("TOP 10 BATTERS BY BASELINE-ADJUSTED UPLIFT")
print("═"*70)
print(bt_uplift.head(10)[cols].round(3).to_string(index=False))

print("\n" + "═"*70)
print("TOP 10 BOWLERS BY BASELINE-ADJUSTED UPLIFT")
print("═"*70)
print(bo_uplift.head(10)[cols].round(3).to_string(index=False))

bt_uplift.to_csv(OUT/"uplift_batters.csv", index=False)
bo_uplift.to_csv(OUT/"uplift_bowlers.csv", index=False)
gc.collect()


Innings 1: AUC 0.684 | logloss 0.640
Innings 2: AUC 0.889 | logloss 0.418
Pressure balls: 49,746 (incl. 38 Super Over balls) | Normal balls: 149,136

Bootstrapping batters...
  boot 50/300
  boot 100/300
  boot 150/300
  boot 200/300
  boot 250/300
  boot 300/300
Bootstrapping bowlers...
  boot 50/300
  boot 100/300
  boot 150/300
  boot 200/300
  boot 250/300
  boot 300/300

Eligible batters: 130 | bowlers: 124

══════════════════════════════════════════════════════════════════════
TOP 10 BATTERS BY BASELINE-ADJUSTED UPLIFT
══════════════════════════════════════════════════════════════════════
            name     n     nn     P      N  Delta  post_delta   lo90  hi90  boot_top10_pct
 Ashutosh Sharma 115.0  181.0 1.959 -0.004  1.959       0.828  0.413 1.243            59.0
       Tim David 459.0  718.0 0.928  0.188  0.854       0.636  0.361 0.912            66.0
     Rinku Singh 380.0  881.0 0.881  0.036  0.866       0.614  0.320 0.909            58.0
     Rashid Khan 171.0  322.0 1.18

0

# **Leakage Tests**

In [15]:
# ══════════════════════════════════════════════════════════════════════════
# LEAKAGE TEST — In-Sample vs Out-of-Fold Skill Rankings (Final Pipeline)
# ══════════════════════════════════════════════════════════════════════════
import pandas as pd, numpy as np, gc
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier as HGB
from scipy.sparse import csr_matrix, hstack

# 1. Setup F (legal balls)
st = pd.read_pickle(OUT / "state.pkl")
st["phase"] = pd.cut(st["overid"], [0, 6, 15, 20], labels=[0, 1, 2]).astype(int)
if "season" not in st.columns:
    st = st.merge(pd.read_csv(DATA/"competitions.csv", usecols=["comp_id","season"]), on="comp_id", how="left")
F = st[st["legal_ball"] & st["bat_team_won"].notna()].copy()
F["y"] = F["runs_bat"].astype(float)
F["season_c"] = F["season"].astype(str)

striker_ids = sorted(F["striker_id"].unique())
bowler_ids  = sorted(F["bowler_id"].unique())
seasons     = sorted(F["season_c"].unique())
venues      = sorted(F["venue_id"].dropna().astype(int).unique())
sid = {v: i for i, v in enumerate(striker_ids)}
bid = {v: i for i, v in enumerate(bowler_ids)}

def design(frame):
    n = len(frame)
    def onehot(series, mapping):
        cols = series.map(mapping).fillna(0).astype(int).values
        return csr_matrix((np.ones(n, dtype=np.float32), (np.arange(n), cols)), shape=(n, len(mapping)))
    S = onehot(frame["striker_id"], sid); B = onehot(frame["bowler_id"], bid)
    P = pd.get_dummies(frame["phase"].astype(int), prefix="p").reindex(columns=[f"p_{i}" for i in range(3)], fill_value=0).values
    Se = pd.get_dummies(frame["season_c"], prefix="s").reindex(columns=[f"s_{s}" for s in seasons], fill_value=0).values
    V = pd.get_dummies(frame["venue_id"].astype(int), prefix="v").reindex(columns=[f"v_{v}" for v in venues], fill_value=0).values
    return hstack([S, B, csr_matrix(P), csr_matrix(Se), csr_matrix(V)]).tocsr()

X_all = design(F)
y = F["y"].values
groups = F["match_id"].values
best_alpha = 50.0

# ── A. In-Sample Skills ──
mdl_in = Ridge(alpha=best_alpha, fit_intercept=True, solver="lsqr").fit(X_all, y)
c_in = mdl_in.coef_
b_in = pd.Series(c_in[:len(striker_ids)], index=striker_ids); b_in -= b_in.mean()
w_in = pd.Series(c_in[len(striker_ids):len(striker_ids)+len(bowler_ids)], index=bowler_ids); w_in -= w_in.mean()

# ── B. Out-of-Fold Skills ──
F["bat_skill_oof"] = 0.0
F["bowl_skill_oof"] = 0.0
for tr, te in GroupKFold(4).split(X_all, y, groups):
    m = Ridge(alpha=best_alpha, fit_intercept=True, solver="lsqr").fit(X_all[tr], y[tr])
    c = m.coef_
    b_s = pd.Series(c[:len(striker_ids)], index=striker_ids); b_s -= b_s.mean()
    w_s = pd.Series(c[len(striker_ids):len(striker_ids)+len(bowler_ids)], index=bowler_ids); w_s -= w_s.mean()
    F.loc[F.index[te], "bat_skill_oof"] = F.iloc[te]["striker_id"].map(b_s).fillna(0.0).values
    F.loc[F.index[te], "bowl_skill_oof"] = F.iloc[te]["bowler_id"].map(w_s).fillna(0.0).values

# ── C. Pipeline Function ──
def run_pipeline(mode):
    df_wp = st.copy()
    df_wp["phase"] = pd.cut(df_wp["overid"], [0, 6, 15, 20], labels=[0, 1, 2]).astype(int)
    if mode == "in_sample":
        df_wp["batter_skill"] = df_wp["striker_id"].map(b_in).fillna(0.0)
        df_wp["bowler_skill"] = df_wp["bowler_id"].map(w_in).fillna(0.0)
    else:
        df_wp["batter_skill"] = 0.0; df_wp["bowler_skill"] = 0.0
        df_wp.loc[F.index, "batter_skill"] = F["bat_skill_oof"]
        df_wp.loc[F.index, "bowler_skill"] = F["bowl_skill_oof"]

    d0 = df_wp[(df_wp["balls_left_before"] > 0) & df_wp["bat_team_won"].notna()].copy()
    d0["pk"] = d0["is_t20i"].astype(int) * 3 + d0["phase"]
    lg = d0[d0["legal_ball"]]
    oc = np.where(lg["is_wicket"].astype(bool), "W", lg["runs_bat"].clip(upper=6).replace({5: 4}).astype(str))
    P_OUT = lg.assign(o=oc).groupby("pk")["o"].value_counts(normalize=True).unstack().fillna(0)
    OUTS = [("W", 0, 1)] + [(str(r), r, 0) for r in [0, 1, 2, 3, 4, 6]]

    CFG = {
        1: dict(feat=["score_before", "wkts_before", "balls_left_before", "is_t20i", "batter_skill", "bowler_skill"],
                mk=lambda: HGB(max_depth=2, max_iter=120, learning_rate=0.05, random_state=42)),
        2: dict(feat=["runs_needed_before", "wkts_before", "balls_left_before", "is_t20i", "batter_skill", "bowler_skill"],
                mk=lambda: HGB(max_iter=250, learning_rate=0.06, max_leaf_nodes=15, min_samples_leaf=200, l2_regularization=1.0, random_state=0)),
    }

    def shift_state(X, i, r, w, bl_drop):
        N = X.copy()
        if i == 1: N[:, 0] += r
        else:      N[:, 0] -= r
        N[:, 1] += w; N[:, 2] = np.clip(N[:, 2] - bl_drop, 0, None)
        return N
    def terminal(wp, N, i):
        if i == 2:
            wp = np.where(N[:, 0] <= 0, 1.0, wp)
            wp = np.where((N[:, 1] >= 10) | ((N[:, 2] <= 0) & (N[:, 0] > 0)), 0.0, wp)
        return wp

    wp_all = []
    for i in (1, 2):
        d = d0[d0["innings"] == i].copy()
        X = d[CFG[i]["feat"]].astype(float).values; yw = d["bat_team_won"].astype(int).values
        wp_b, wp_a, lev, edw = (np.zeros(len(d)) for _ in range(4))
        for tr, te in GroupKFold(5).split(X, yw, d["match_id"]):
            mdl = CFG[i]["mk"]().fit(X[tr], yw[tr]); Xt = X[te]; T = d.iloc[te]
            p0 = mdl.predict_proba(Xt)[:, 1]
            Na = shift_state(Xt, i, T["approx_runs"].values, T["is_wicket"].astype(int).values, T["legal_ball"].astype(int).values)
            wp_a[te] = terminal(mdl.predict_proba(Na)[:, 1], Na, i)
            L = np.zeros(len(te)); E = np.zeros(len(te))
            for name, r, w in OUTS:
                Nh = shift_state(Xt, i, r, w, 1); p1 = terminal(mdl.predict_proba(Nh)[:, 1], Nh, i)
                pr = T["pk"].map(P_OUT[name]).values
                L += pr * np.abs(p1 - p0); E += pr * (p1 - p0)
            wp_b[te], lev[te], edw[te] = p0, L, E
        d["wp_before"], d["wp_after"], d["leverage"], d["exp_dwp"] = wp_b, wp_a, lev, edw
        wp_all.append(d)
    
    wp_new = pd.concat(wp_all, ignore_index=True)
    wp_new["is_pressure"] = False
    for inn in (1, 2):
        mask = wp_new["innings"] == inn
        wp_new.loc[mask, "is_pressure"] = wp_new.loc[mask, "leverage"] >= wp_new.loc[mask, "leverage"].quantile(0.75)

    wv = wp_new.copy()
    FULL = {"India", "Australia", "England", "South Africa", "New Zealand", "Pakistan", "Sri Lanka", "Bangladesh", "West Indies", "Afghanistan", "Zimbabwe", "Ireland"}
    _mm = pd.read_csv(DATA/"matches.csv", usecols=["match_id","comp_id"]).merge(pd.read_csv(DATA/"competitions.csv", usecols=["comp_id","comp_name"]), on="comp_id")
    _tn = pd.read_csv(DATA/"teams.csv").drop_duplicates("team_id").set_index("team_id")["team_name"]
    _t = wv.drop_duplicates("match_id")[["match_id","batting_team_id","bowling_team_id"]].merge(_mm, on="match_id")
    _t["ipl"] = _t.comp_name.str.startswith("IPL"); _t["minor"] = _t.comp_name.str.contains("Emerging|Rising")
    _t["both_full"] = _t.batting_team_id.map(_tn).isin(FULL) & _t.bowling_team_id.map(_tn).isin(FULL)
    _keep = set(_t.loc[_t.ipl | (~_t.minor & _t.both_full), "match_id"])
    wv = wv[wv.match_id.isin(_keep)].copy()

    wv["dwp"] = (wv["wp_after"] - wv["wp_before"]) * 100
    wv["excess"] = wv["dwp"] - wv["exp_dwp"] * 100
    neutral = wv["extras_type"].fillna("none").isin(["byes", "legbyes"])
    
    bat0 = wv[wv["legal_ball"] & ~neutral].copy()
    bat0["x"] = bat0["excess"]
    bat0.loc[bat0["is_wicket"].astype(bool) & ~bat0["striker_out"].astype(bool), "x"] = 0.0
    bat0["pid"] = bat0["striker_id"]
    
    bow0 = wv[~neutral].copy()
    bow0["x"] = -bow0["excess"]
    bow0.loc[bow0["is_wicket"].astype(bool) & ~bow0["bowler_wkt"].astype(bool), "x"] = 0.0
    bow0["pid"] = bow0["bowler_id"]

    def label(q): return wv.groupby("innings")["leverage"].transform(lambda s: s >= s.quantile(q))
    def prep(frame, lab):
        f = frame.assign(is_p=lab.loc[frame.index].values)
        p, n = f[f.is_p], f[~f.is_p]
        pm = p.groupby(["match_id", "pid"])["x"].agg(n="size", sx="sum").reset_index()
        nm = n.groupby(["match_id", "pid"])["x"].agg(nn="size", snx="sum").reset_index()
        return pm, nm, p["x"].var()
    def agg_players(pm, nm, wt=None):
        if wt is not None:
            fp = pm["match_id"].map(wt).fillna(0).values; fn = nm["match_id"].map(wt).fillna(0).values
            pm = pm.assign(n=pm.n * fp, sx=pm.sx * fp); nm = nm.assign(nn=nm.nn * fn, snx=nm.snx * fn)
        t = pm.groupby("pid")[["n", "sx"]].sum().join(nm.groupby("pid")[["nn", "snx"]].sum(), how="left").fillna(0)
        t = t[t.n > 0].copy(); t["P"] = t.sx / t.n
        t["N"] = np.where(t.nn > 0, t.snx / t.nn.replace(0, np.nan), np.nan)
        return t
    def eb_fit(t, sig2, model, minn=15):
        t = t.copy(); t["s2"] = sig2 / t.n; e = t[(t.n >= minn) & t.N.notna()]
        tau2 = max(e.P.var() - e.s2.mean(), 1e-9)
        for _ in range(5):
            wt = 1 / (tau2 + e.s2)
            if model == "M1":
                sw = np.sqrt(wt.values); a, bb = np.linalg.lstsq(np.c_[np.ones(len(e)), e.N] * sw[:, None], e.P.values * sw, rcond=None)[0]
            else:
                a, bb = np.average(e.P, weights=wt), 0.0
            tau2 = max(np.average((e.P - (a + bb * e.N)) ** 2 - e.s2, weights=wt), 1e-9)
        t["prior"] = a + bb * t.N.fillna(t.N.mean()) if model == "M1" else a
        t["B"] = tau2 / (tau2 + t.s2); t["post"] = t.prior + t.B * (t.P - t.prior); t["psd"] = np.sqrt(t.B * t.s2)
        return t
    def run(frame, namecol, q=0.75, model="M1"):
        pm, nm, sig2 = prep(frame, label(q))
        T = eb_fit(agg_players(pm, nm), sig2, model)
        T = T[T.n >= 100].copy()
        T["name"] = frame.groupby("pid")[namecol].agg(lambda s: s.mode()[0]).reindex(T.index)
        return dict(T=T.sort_values("post", ascending=False))
    
    return set(run(bat0, "striker_name")["T"].head(10)["name"]), set(run(bow0, "bowler_name")["T"].head(10)["name"])

print("Running In-Sample Pipeline (Leaky)...")
bat_in_set, bowl_in_set = run_pipeline("in_sample")
print("  Batters :", sorted(bat_in_set))
print("  Bowlers :", sorted(bowl_in_set))

print("\nRunning Out-of-Fold Pipeline (Correct)...")
bat_oof_set, bowl_oof_set = run_pipeline("oof")
print("  Batters :", sorted(bat_oof_set))
print("  Bowlers :", sorted(bowl_oof_set))

print("\n── COMPARISON ──")
print(f"Batters overlap  : {len(bat_in_set & bat_oof_set)}/10")
print(f"Bowlers overlap  : {len(bowl_in_set & bowl_oof_set)}/10")
print("Batters only in In-Sample:", sorted(bat_in_set - bat_oof_set))
print("Batters only in OOF     :", sorted(bat_oof_set - bat_in_set))
print("Bowlers only in In-Sample:", sorted(bowl_in_set - bowl_oof_set))
print("Bowlers only in OOF     :", sorted(bowl_oof_set - bowl_in_set))

Running In-Sample Pipeline (Leaky)...
  Batters : ['Ashutosh Sharma', 'Dasun Shanaka', 'Dinesh Karthik', 'Heinrich Klaasen', 'Mahendra Singh Dhoni', 'Namboori Tilak Varma', 'Rinku Singh', 'Shashank Singh', 'Suryakumar Yadav', 'Tim David']
  Bowlers : ['Akeal Hosein', 'Jimmy Neesham', 'Josh Hazlewood', 'Mohammad Nawaz', 'Mohsin Khan', 'Rashid Khan', 'Sunil Narine', 'Umran Malik', 'Varun Chakravarthy', 'Wanindu Hasaranga']

Running Out-of-Fold Pipeline (Correct)...
  Batters : ['Ashutosh Sharma', 'Dasun Shanaka', 'Heinrich Klaasen', 'Kamindu Mendis', 'Mohammad Nawaz', 'Namboori Tilak Varma', 'Reeza Hendricks', 'Rinku Singh', 'Suryakumar Yadav', 'Tim David']
  Bowlers : ['Corbin Bosch', 'Jasprit Bumrah', 'Jimmy Neesham', 'Josh Hazlewood', 'Mohammad Nawaz', 'Rashid Khan', 'Sunil Narine', 'Umran Malik', 'Varun Chakravarthy', 'Wanindu Hasaranga']

── COMPARISON ──
Batters overlap  : 7/10
Bowlers overlap  : 8/10
Batters only in In-Sample: ['Dinesh Karthik', 'Mahendra Singh Dhoni', 'Shashank S

# **Validation**

In [16]:
# ══════════════════════════════════════════════════════════════════════════
# FINAL VALIDATION — Sensitivity & Robustness on Uplift (P - N) Metric
# ══════════════════════════════════════════════════════════════════════════
import gc, numpy as np, pandas as pd
from scipy.stats import spearmanr

# Helper to evaluate Uplift with custom thresholds
def evaluate_uplift(frame, namecol, q=0.75, mp=100, mn=100):
    lab_map = wv.groupby("innings")["leverage"].quantile(q).to_dict()
    f = frame.copy()
    f["is_p"] = f["leverage"].values >= f["innings"].map(lab_map).values
    T = build_eb(f)
    T = T[(T.n >= mp) & (T.nn >= mn)].copy()
    T["name"] = f.groupby("pid")[namecol].agg(lambda s: s.mode()[0]).reindex(T.index)
    return T.sort_values("post_delta", ascending=False).reset_index()

base_bat = set(bt_uplift.head(10)["name"])
base_bowl = set(bo_uplift.head(10)["name"])

print("═" * 72)
print("TEST 1: PRESSURE THRESHOLD SWEEP (Uplift Metric)")
print("═" * 72)
for q in (0.70, 0.80, 0.85):
    tb = set(evaluate_uplift(bat0, "striker_name", q=q).head(10)["name"])
    tw = set(evaluate_uplift(bow0, "bowler_name", q=q).head(10)["name"])
    print(f"  Top {round((1-q)*100)}% pressure (q={q}): batters {len(tb & base_bat)}/10 | bowlers {len(tw & base_bowl)}/10")

print("\n" + "═" * 72)
print("TEST 2: MINIMUM BALLS SWEEP (Uplift Metric)")
print("═" * 72)
for mp in (60, 150, 200):
    tb = set(evaluate_uplift(bat0, "striker_name", mp=mp, mn=mp).head(10)["name"])
    tw = set(evaluate_uplift(bow0, "bowler_name", mp=mp, mn=mp).head(10)["name"])
    print(f"  Min {mp} pressure & {mp} normal balls: batters {len(tb & base_bat)}/10 | bowlers {len(tw & base_bowl)}/10")

print("\n" + "═" * 72)
print("TEST 3: TIME SPLIT — 2022-24 vs 2025-26 (Uplift Metric)")
print("═" * 72)
st_dates = pd.read_csv(DATA/"matches.csv", usecols=["match_id","match_date"])
st_dates["year"] = pd.to_datetime(st_dates["match_date"]).dt.year
bat0["year"] = bat0["match_id"].map(st_dates.set_index("match_id")["year"])
bow0["year"] = bow0["match_id"].map(st_dates.set_index("match_id")["year"])

early_b = evaluate_uplift(bat0[bat0["year"] < 2025], "striker_name", mp=50, mn=50)
late_b  = evaluate_uplift(bat0[bat0["year"] >= 2025], "striker_name", mp=50, mn=50)
early_w = evaluate_uplift(bow0[bow0["year"] < 2025], "bowler_name", mp=50, mn=50)
late_w  = evaluate_uplift(bow0[bow0["year"] >= 2025], "bowler_name", mp=50, mn=50)

j_b = early_b[["name","post_delta"]].merge(late_b[["name","post_delta"]], on="name", suffixes=("_e","_l"))
j_w = early_w[["name","post_delta"]].merge(late_w[["name","post_delta"]], on="name", suffixes=("_e","_l"))
rho_b = j_b["post_delta_e"].corr(j_b["post_delta_l"], method="spearman") if len(j_b)>5 else np.nan
rho_w = j_w["post_delta_e"].corr(j_w["post_delta_l"], method="spearman") if len(j_w)>5 else np.nan

print(f"  Batters Spearman (early vs late): {rho_b:+.3f}")
print(f"  Bowlers Spearman (early vs late): {rho_w:+.3f}")

print("\n" + "═" * 72)
print("TEST 4: CLUTCH VS TALENT CORRELATION")
print("═" * 72)
for k, T in (("Batters", bt_uplift), ("Bowlers", bo_uplift)):
    mask = T["P"].notna() & T["N"].notna()
    rho, _ = spearmanr(T.loc[mask, "P"], T.loc[mask, "N"])
    print(f"  {k}: Spearman(Pressure, Normal) = {rho:+.3f}")

print("\n✓ Final validation complete. Ready for the report.")

════════════════════════════════════════════════════════════════════════
TEST 1: PRESSURE THRESHOLD SWEEP (Uplift Metric)
════════════════════════════════════════════════════════════════════════
  Top 30% pressure (q=0.7): batters 8/10 | bowlers 7/10
  Top 20% pressure (q=0.8): batters 7/10 | bowlers 8/10
  Top 15% pressure (q=0.85): batters 6/10 | bowlers 6/10

════════════════════════════════════════════════════════════════════════
TEST 2: MINIMUM BALLS SWEEP (Uplift Metric)
════════════════════════════════════════════════════════════════════════
  Min 60 pressure & 60 normal balls: batters 8/10 | bowlers 8/10
  Min 150 pressure & 150 normal balls: batters 8/10 | bowlers 6/10
  Min 200 pressure & 200 normal balls: batters 5/10 | bowlers 5/10

════════════════════════════════════════════════════════════════════════
TEST 3: TIME SPLIT — 2022-24 vs 2025-26 (Uplift Metric)
════════════════════════════════════════════════════════════════════════
  Batters Spearman (early vs late): -0.022


In [17]:
# ══════════════════════════════════════════════════════════════════════════
# SLIDING SCALE SPLIT-HALF (Testing Cutoffs: 150, 200, 250)
# ══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

bat0["half"] = np.where(bat0["match_id"] % 2 == 0, "even", "odd")
bow0["half"] = np.where(bow0["match_id"] % 2 == 0, "even", "odd")

def test_cutoff(cutoff, role, frame, namecol):
    def eval_half(half):
        T = build_eb(frame[frame["half"] == half])
        T = T[(T.n >= cutoff) & (T.nn >= cutoff)].copy()
        T["name"] = frame.groupby("pid")[namecol].agg(lambda s: s.mode()[0]).reindex(T.index)
        return T.sort_values("post_delta", ascending=False).reset_index()
    
    even = eval_half("even")
    odd  = eval_half("odd")
    merged = even[["name", "post_delta"]].merge(odd[["name", "post_delta"]], on="name", suffixes=("_e", "_o"))
    
    if len(merged) < 3:
        print(f"  Cutoff {cutoff}: Only {len(merged)} players qualify. (Too small to test)")
        return
        
    rho, _ = spearmanr(merged["post_delta_e"], merged["post_delta_o"])
    top_e = set(even.head(10)["name"])
    top_o = set(odd.head(10)["name"])
    overlap = len(top_e & top_o)
    print(f"  Cutoff {cutoff}: n={len(merged):<3} | Spearman {rho:+.3f} | Top-10 overlap: {overlap}/10")

print("── BATTERS ──")
for c in [150, 200, 250, 300]:
    test_cutoff(c, "batters", bat0, "striker_name")

print("\n── BOWLERS ──")
for c in [150, 200, 250, 300]:
    test_cutoff(c, "bowlers", bow0, "bowler_name")

── BATTERS ──
  Cutoff 150: n=28  | Spearman +0.248 | Top-10 overlap: 3/10
  Cutoff 200: n=12  | Spearman -0.133 | Top-10 overlap: 2/10
  Cutoff 250: n=7   | Spearman +0.500 | Top-10 overlap: 6/10
  Cutoff 300: n=3   | Spearman +0.500 | Top-10 overlap: 3/10

── BOWLERS ──
  Cutoff 150: n=30  | Spearman +0.175 | Top-10 overlap: 3/10
  Cutoff 200: n=15  | Spearman +0.146 | Top-10 overlap: 5/10
  Cutoff 250: n=9   | Spearman +0.383 | Top-10 overlap: 7/10
  Cutoff 300: n=4   | Spearman +1.000 | Top-10 overlap: 4/10
